## External Validation Dataset Construction (PATSTAT HAN Sample)

**Purpose.** To test whether the harmonisation pipeline generalises beyond the initial gold standard without re-tuning any parameters, this script builds an independent validation dataset directly from PATSTAT (`tls206_person` joined with `tls207_pers_appln`) via the EPO TIP BigQuery interface.

**Ground-truth reference (`han_id`).** The benchmark labels come from PATSTAT's OECD Harmonised Applicant Names (`han_id`), a standard reference grouping in patent economics. Because `han_id` is generated primarily through rule-based string standardisation rather than manual curation, it tends to be conservative (occasionally keeping subsidiaries or spelling variants separate that semantic embeddings merge), making it a strict out-of-sample test of precision and threshold stability.

**Sampling & filtering criteria:**
1. **Corporate applicants only:** Filters to `applt_seq_nr > 0` (excluding individual inventors) and `psn_sector = 'COMPANY'` to match the target scope of the baseline pipeline[cite: 2].
2. **Valid, non-trivial clusters:** Excludes missing/zero `han_id` values and singletons (`COUNT(DISTINCT person_name) = 1`), retaining groups with **2 to 150 distinct raw spellings** so evaluation metrics reflect genuine name-matching decisions rather than trivial singleton preservation.
3. **Balanced sample size:** Randomly draws **5000 `han_id` groups** (`ORDER BY RAND() LIMIT 5000`) and exports all associated `(Group, han_name, comp, person_ctry_code)` records to `PATSTAT_HAN_5000_val.csv`, matching the scale and schema of the primary gold standard (`PATSTAT_500_label_datav4.csv`)[cite: 2].

*Note: Once generated, `PATSTAT_HAN_5000_val.csv` is kept frozen on disk so that all subsequent validation runs remain strictly reproducible.*

In [1]:
import pandas as pd
from epo.tipdata.patstat import PatstatClient

# 1. Connect to PATSTAT on TIP
patstat = PatstatClient(env="PROD")

# 2. Query 5000 multi-variant corporate HAN groups (BigQuery / Standard SQL dialect used on TIP)
query = """
#standardSQL
WITH multi_variant_han AS (
    SELECT 
        p.han_id,
        MAX(p.han_name) AS han_name,
        COUNT(DISTINCT p.person_name) AS n_variants
    FROM tls206_person p
    JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
    WHERE pa.applt_seq_nr > 0
      AND p.psn_sector = 'COMPANY'
      AND p.han_id IS NOT NULL
      AND p.han_id > 0
      AND p.person_name IS NOT NULL
      AND TRIM(p.person_name) <> ''
    GROUP BY p.han_id
    HAVING COUNT(DISTINCT p.person_name) BETWEEN 2 AND 150
),
sampled_groups AS (
    SELECT han_id, han_name
    FROM multi_variant_han
    ORDER BY RAND()
    LIMIT 5000
)
SELECT DISTINCT
    sg.han_id AS `Group`,
    sg.han_name,
    p.person_name AS comp,
    p.person_ctry_code
FROM sampled_groups sg
JOIN tls206_person p ON sg.han_id = p.han_id
JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
WHERE pa.applt_seq_nr > 0
  AND p.person_name IS NOT NULL
  AND TRIM(p.person_name) <> ''
"""

# 3. Execute and save directly to your TIP workspace directory
res = patstat.sql_query(query, use_legacy_sql=False)
df_val = pd.DataFrame([dict(r) for r in res]) # handles BigQuery Row objects or dicts cleanly

df_val.to_csv("PATSTAT_HAN_5000_val.csv", index=False)

print(f"Saved {len(df_val):,} rows across {df_val['Group'].nunique():,} HAN groups.")
df_val.head()

Saved 17,013 rows across 5,000 HAN groups.


,Group,han_name,comp,person_ctry_code
0,154,1 800 CONTACTS INC,"1-800 Contacts, Inc.",US
1,154,1 800 CONTACTS INC,"1-800 CONTACTS, INC.",US
2,154,1 800 CONTACTS INC,1 800 CONTACTS INC.,US
3,154,1 800 CONTACTS INC,1 800 CONTACTS INC,US
4,154,1 800 CONTACTS INC,"1-800- CONTACTS, INC.",US
